# Giai đoạn 2–3 — Chuẩn bị, chia Train/Test và Dummy baseline

Theo kế hoạch dự án v3.1. Nhãn `recommended_career` mô tả **nhóm nghề nghiệp được khuyến nghị**, không phải tỷ lệ trúng tuyển đại học.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("Project:", ROOT)

## 1. X, y, LabelEncoder và chia đúng 80/20. Test được giữ riêng.

In [ ]:
from career_ml.data import load_data,prepare_xy,split_data,input_schema
from career_ml.modeling import make_pipeline,make_cv,feature_names_after_fs
from sklearn.model_selection import cross_validate
X,y,encoder=prepare_xy(load_data())
X_train,X_test,y_train,y_test=split_data(X,y)
print('Train:',X_train.shape,'Test:',X_test.shape)
print('Labels:',encoder.classes_.tolist())
assert not set(X_train.index) & set(X_test.index)
print('Phân bố Train:'); print(y_train.value_counts().sort_index())

## 2. Kiểm tra pipeline (preprocess + selector + classifier) khi fit *chỉ Train*

In [ ]:
pipe=make_pipeline(X_train,"Logistic Regression","Filter")
pipe.fit(X_train,y_train)
print('Số đặc trưng được chọn:',len(feature_names_after_fs(pipe)))
print('Shape dự đoán:',pipe.predict(X_train.iloc[:3]).shape)
print('Shape xác suất:',pipe.predict_proba(X_train.iloc[:3]).shape)
# Không chạm tới Test trong cell này.

## 3. Baseline ngẫu nhiên dùng 5-fold trên Train

In [ ]:
baseline=make_pipeline(X_train,"Dummy","None")
cv_score=cross_validate(baseline,X_train,y_train,cv=make_cv(),
                        scoring={"accuracy":"accuracy","f1_macro":"f1_macro"},n_jobs=2)
print('Dummy Accuracy CV:',cv_score['test_accuracy'].mean())
print('Dummy F1 Macro CV:',cv_score['test_f1_macro'].mean())
print('Điểm này là Train CV, KHÔNG phải Test.')

**Ghi chú:** Đây là mô hình nền; 12 GridSearchCV chính nằm trong notebook 04, mỗi bộ chọn nằm trong Pipeline được fit lại từng fold.